# Lexical retrieval

"Lexical" means "word-related". Lexical retrieval finds chunks that share important
words, phrases, or software identifiers with the question.

[Tutorial 70](./70_semantic_retrieval.ipynb) searched by meaning using embeddings. This tutorial searches by wording using TF-IDF (Term Frequency-Inverse Document Frequency). No embedding model is required.

This is especially useful for questions containing names such as
`EmbeddingSession`, `lexical_terms`, or `build_lexical_index`.


In [ ]:
from pathlib import Path

root = Path.cwd()
if not (root / "qachat").is_dir():
    root = root.parent

print(f"QAChat root: {root}")


In [ ]:
import sys

if str(root) not in sys.path:
    sys.path.insert(0, str(root))


In [ ]:
from setup.notebook import prepare_notebook

root = prepare_notebook(root)


## 1. Prepare repository chunks

These are the same staps as in [Tutorial 50](./50_source_structure.ipynb) and [Tutorial 60](./60_embeddings.ipynb). It selects real QAChat files, reads their natural units, and makes chunks.

Lexical retrieval does not need a model tokenizer, so this example uses a
visible word counter only to keep large source units bounded.


In [ ]:
from qachat import scan_repository


report = scan_repository(
    root,
    extensions=[".py"],
    include_paths=["qachat"],
)

print(f"Selected {len(report['selected_files'])} files:")

for i, file in enumerate(report["selected_files"]):
    print(f"{i}. {file}")


In [ ]:
from qachat import extract_source_units, read_repository_files


source_files = read_repository_files(report)
units = extract_source_units(source_files)

print(f"Read {len(source_files)} files")
print(f"Natural units: {len(units)}")


In [ ]:
def count_words(text: str) -> int:
    return len(text.split())


In [ ]:
from qachat import chunk_source_units


chunks = chunk_source_units(
    units,
    count_tokens=count_words,
    target_tokens=120,
    max_tokens=160,
    overlap_tokens=20,
)

print(f"Searchable chunks: {len(chunks)}")


## 2. Make one-word and two-word terms

A "unigram" is one searchable word. A "bigram" is two adjacent words.
Bigrams preserve short phrases such as "semantic search".

Software names are divided too. For example, `scanRepository` and
`scan_repository` both contribute the words "scan" and "repository".


In [ ]:
from qachat import lexical_terms


example = "scanRepository selects source files"

terms = lexical_terms(example)

print(terms)


In [ ]:
unigrams = [term for term in terms if " " not in term]

print("One-word terms:", unigrams)


In [ ]:
bigrams = [term for term in terms if " " in term]

print("Two-word terms:", bigrams)


## 3. Build the lexical index

Each chunk is converted into searchable terms.

- **TF** (Term Frequency) gives more weight to terms that appear often in one chunk.
- **IDF** (Inverse Document Frequency) gives more weight to terms that are rare across the repository.
- **TF-IDF** combines both into a sparse vector for each chunk.

The lexical index stores these vectors so they can be reused for later
questions.


In [ ]:
from qachat import build_lexical_index


lexical_index = build_lexical_index(chunks)

assert len(lexical_index.chunks) == len(chunks)
assert len(lexical_index.vectors) == len(chunks)

print(f"Indexed chunks: {len(lexical_index.chunks)}")
print(f"Known terms: {len(lexical_index.inverse_document_frequency)}")


In [ ]:
first_vector = lexical_index.vectors[0]
strongest_terms = sorted(
    first_vector.items(),
    key=lambda item: item[1],
    reverse=True,
)

print("Terms from strongest:")
for term in strongest_terms:
    print(term)


## 4. Search by wording

The question remains plain text. It is converted into a TF-IDF vector using
the same vocabulary and IDF weights as indexes.

Cosine similarity then compares the question with each chunk and ranks the
strongest word and phrase matches first.


In [ ]:
QUESTION = (
    "Which function turns software identifiers into one-word "
    "and two-word terms?"
)

print(QUESTION)


In [ ]:
from qachat import lexical_search


results = lexical_search(
    lexical_index,
    QUESTION,
    count=5,
)

results


In [ ]:
scores = [result.score for result in results]

assert results
assert scores == sorted(scores, reverse=True)

print(f"Retrieved chunks: {len(results)}")
print(f"Highest score: {scores[0]}")


## 5. Inspect the ranking

A higher score means the query and chunk share more important terms. As with
semantic scores, this orders candidates; it is not a probability that the
chunk is correct.


In [ ]:
for rank, result in enumerate(results, start=1):
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    name = chunk.symbol or chunk.section or "(module context)"
    print(f"{rank}. {result.score:.3f} | {location} | {name}")


In [ ]:
best = results[0]

print("Most relevant chunk:\n")
print(best.chunk.text)


In [ ]:
for rank, result in enumerate(results):
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    name = chunk.symbol or chunk.section or "(module context)"
    print(
        f"\n>>> Chunk {rank}. {result.score:.3f} | {location} | {name} <<<\n"
    )
    print(f"{result.chunk.text}")


## 6. Search an exact identifier

Exact identifiers are a lexical strength. The query below directly names a
function, so its source chunk should rank strongly without an embedding model.


In [ ]:
identifier_results = lexical_search(
    lexical_index,
    "build_lexical_index",
    count=3,
)

for rank, result in enumerate(identifier_results):
    chunk = result.chunk
    location = f"{chunk.path}:{chunk.start_line}-{chunk.end_line}"
    name = chunk.symbol or chunk.section or "(module context)"
    print(
        f"\n>>> Chunk {rank}. {result.score:.3f} | {location} | {name} <<<\n"
    )
    print(result.chunk.text)
